# Hammurab.AI — Qwen2.5-7B + Unsloth QLoRA (Free Colab T4)
**CIF425 Term Project**

**Free Colab kısıtları:**
- Max 12 saat / 90 dk idle disconnect
- Tab kapanırsa eğitim ölür

**Bu notebook'un stratejisi:**
1. Checkpoint'leri Google Drive'a yazar → disconnect olsa bile kayıp az
2. Keep-alive snippet ile idle disconnect'i geciktirir
3. **1 epoch default** — ~1.5-2 saat. Sonuç iyiyse `HAMMURAB_EPOCHS=3` ile tekrar çalıştır

**Kullanım:** Runtime > Change runtime type > **T4 GPU** seç, sonra hücreleri sırayla çalıştır.

## 1. Repo + Bağımlılıklar

In [ ]:
!git clone https://github.com/Alp33er/hammurab.ai.git
%cd hammurab.ai
!git checkout development

In [ ]:
# Unsloth + QLoRA stack — kurulum 3-5 dk sürer
!pip install -q -r training/requirements.txt

## 2. GPU Kontrol

T4 (16 GB) bekleniyor. Eğer GPU yoksa: Runtime > Change runtime type > T4 GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU YOK — Runtime > Change runtime type > GPU seç!"
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16: {torch.cuda.is_bf16_supported()}  (T4'te False bekleniyor)")

## 3. Google Drive'ı Bağla — Checkpoint'ler Buraya Yazılacak

Free Colab disconnect ettiğinde RAM/disk uçar ama Drive kalır. Bu yüzden checkpoint'leri Drive'a yazıyoruz; disconnect sonrası `HAMMURAB_RESUME=1` ile devam edebilirsin.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
OUTPUT_DIR = "/content/drive/MyDrive/hammurab_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["HAMMURAB_OUTPUT_DIR"] = OUTPUT_DIR
print(f"Output: {OUTPUT_DIR}")

## 4. Keep-Alive (Idle Disconnect'i Geciktirir)

**Önemli:** Browser tab'ını **AÇIK** tut. Bu hücreyi çalıştırınca her 60 sn'de boş bir tıklama simüle eder, idle timer sıfırlanır.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    console.log("Keep-alive: " + new Date().toLocaleString());
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print("Keep-alive aktif. Tab'ı kapatma!")

## 5. Veri Seti Oluştur

Kanun JSON'larından Qwen ChatML formatında ~18K Q&A çifti üretir.

In [ ]:
!python training/prepare_dataset.py

In [ ]:
# Veri seti örneği
import json
with open("training/data/hukuk_qa.jsonl", "r") as f:
    sample = json.loads(f.readline())
for msg in sample["messages"]:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"][:400])
    print()

## 6. Fine-Tuning (1 Epoch — Free Colab Güvenli)

**Konfigürasyon:**
- Qwen2.5-7B-Instruct + 4-bit + LoRA r=16
- max_seq=1024, batch=2, grad_accum=4 → eff. batch=8
- Her 200 adımda Drive'a checkpoint
- Süre tahmini: **~1.5-2 saat** (T4)

**Daha fazla epoch için:** Aşağıdaki hücreyi çalıştırmadan önce `os.environ["HAMMURAB_EPOCHS"] = "3"` ekle (3 epoch ~5-6 saat).

In [ ]:
# Override mümkün — yorum kaldır:
# os.environ["HAMMURAB_EPOCHS"] = "3"
# os.environ["HAMMURAB_MAX_SEQ"] = "2048"  # daha uzun context (yavaşlatır)

!python training/fine_tune.py

## 6b. (Disconnect Olduysa) Checkpoint'ten Devam Et

Eğitim ortasında session koptuysa: yeni runtime aç, hücre 1-5 arasını yeniden çalıştır, sonra **bu hücreyi** çalıştır. Drive'daki son checkpoint'ten devam eder.

In [ ]:
# os.environ["HAMMURAB_RESUME"] = "1"
# !python training/fine_tune.py

## 7. Modeli Test Et

LoRA adapter'ı base model üzerine yükle ve birkaç soru sor.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(model)

SYSTEM = (
    "Sen Türk hukuku konusunda uzman bir hukuk araştırma asistanısın. "
    "Soruları verilen mevzuat metinlerine dayanarak yanıtla, "
    "ilgili kanun maddesini ve referansını mutlaka göster, "
    "context'te olmayan bilgiyi uydurma."
)

def sor(soru, mevzuat=""):
    user_content = soru
    if mevzuat:
        user_content += f"\n\n[İlgili Mevzuat]\n{mevzuat}"
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        inputs, max_new_tokens=512, do_sample=True,
        temperature=0.7, top_p=0.9, repetition_penalty=1.1,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("Model hazır.")

In [ ]:
print(sor("İş kazası durumunda işçinin hakları nelerdir?"))

In [ ]:
print(sor("Türk Borçlar Kanunu madde 49 ne diyor?"))

In [ ]:
print(sor("Kıdem tazminatı nasıl hesaplanır?"))

## 8. Modeli Lokal'e İndir

**LoRA adapter** sadece ~50-100 MB — base model gerekmez. Lokal `app.py` Qwen2.5-7B-Instruct'ı kendi indirir, üzerine bu adapter'ı yükler.

In [ ]:
!zip -r /content/hammurab_lora.zip $OUTPUT_DIR
!ls -lh /content/hammurab_lora.zip

from google.colab import files
files.download("/content/hammurab_lora.zip")

## 9. (Opsiyonel) HuggingFace Hub'a Push

HF'e yüklersen `app.py`'da `from_pretrained("USERNAME/hammurab-qwen-7b-lora")` ile direkt çekersin.

In [ ]:
# from huggingface_hub import login
# login()
# model.push_to_hub("USERNAME/hammurab-qwen-7b-lora")
# tokenizer.push_to_hub("USERNAME/hammurab-qwen-7b-lora")